# Lab: detecting icequakes in raw seismograms

```{note}
This lab accompanies {doc}`cryoseismicity`. The chapter surveys what ice radiates and why; this lab builds the processing chain that turns raw counts on a seismometer into a catalogue of icequake detections. No prior seismology is assumed.
```

Passive seismology is one of the few methods that can hear basal processes directly, not infer them from surface geometry. The workflow from raw data to a detection catalogue is not exotic — it runs on publicly archived data and a few hundred lines of numpy — but each step hides a physics decision. By the end of this lab you should be able to

- locate and fetch raw seismic data from the EarthScope FDSN web service using obspy,
- remove instrument response and bandpass-filter to expose the signal of interest,
- implement the STA/LTA (short-term average over long-term average) detector from scratch and tune it on a real day of data,
- classify detections by frequency content and duration as basal or surface-fracture events,
- articulate what a single-station catalogue can and cannot tell you about source location,
- connect icequake rates to the friction and melt-rate arguments in {doc}`../thermomechanics/basal-motion` and {doc}`elevation`.

The computationally heavy cells are not executed at build time. They are written to be run interactively, and two clearly-marked `# NOT EXECUTED AT BUILD` cells fetch live data. Every other cell can run offline on data saved to disk in those steps.

```{admonition} Patterns from UW-GDA
:class: seealso
The programmatic station-data fetch used here, querying a web service by network, station, and time window rather than downloading an archive, follows the UW Geospatial Data Analysis course of {cite}`shean2023`, which teaches these patterns on SNOTEL station time series whose structure mirrors an FDSN seismic query closely. Module 8 (vector time series, timestamps, and station-data download) covers the pandas datetime handling and web-service query patterns the obspy fetch and STA/LTA detection pipeline here depend on. Work through Module 8 there if the tooling here feels unfamiliar.
```

## Data sources and volumes

The global seismic archive lives at EarthScope (the merged network of the old IRIS and UNAVCO). Its FDSN (International Federation of Digital Seismograph Networks) web service hands you waveforms over HTTP: you send a query specifying a network code, station code, channel code, and time window, and you receive raw counts from the analog-to-digital converter, the integer values that left the seismometer before any physical unit was assigned to them.

Before you query anything, pause on the data-volume arithmetic. A broadband seismometer running at 100 samples per second, three components, 4 bytes per sample, produces roughly **10 MB per station-day**. A network of 20 stations running for a 90-day polar field season produces about 18 GB of raw data. Arrays in Greenland or Antarctica are in that range. The full EarthScope broadband archive runs to the order of **terabytes**. You do not download a season. You fetch a window, the hours around an event of interest, or at most a day or two for statistical work. This is not a computational inconvenience; it is how you must think about the data even when disk space is not the constraint, because a raw continuous seismogram has no natural unit until the instrument response is removed, and carrying gigabytes of raw counts around invites mistakes.

Polar deployments add a further wrinkle. Data from temporary networks — a season of broadband sensors on a surge glacier or an ice-stream sticky spot — are often under a one-to-two-year embargo while the field team does their primary analysis. After that embargo, almost everything in the EarthScope system is **openly archived**, and many productive icequake studies have been done entirely on existing public data. The `# verify:` comments in the next cell give FDSN identifiers for a station that was open as of mid-2025; an instructor can substitute any network and station with open continuous data near ice.

The first honest step after fetching waveforms is **instrument response removal**. The seismometer, its digitizer, and its cables all imprint a frequency-dependent amplitude and phase signature on the raw count record. That signature, stored in the FDSN metadata as a StationXML file, must be deconvolved before the record means anything in physical units — meters per second of ground velocity, or acceleration. The obspy `simulate_seismometer` or `remove_response` method handles this numerically; calling it early, before any filtering, prevents the filter from aliasing the response artifact into the band you care about.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram

# obspy is the standard Python interface to FDSN web services.
# Install with:  pip install obspy
try:
    from obspy import UTCDateTime
    from obspy.clients.fdsn import Client
    OBSPY_AVAILABLE = True
except ImportError:
    OBSPY_AVAILABLE = False
    print('obspy not installed — data-fetch cells will be skipped.')
    print('Run:  pip install obspy   to enable them.')

print('numpy', np.__version__)

## Configuring the fetch

The cell below sets the parameters that the instructor adjusts to point at a current open deployment. The defaults use a broadband station from the Greenland Ice Sheet Monitoring Network (GLISN), network `GL`, which has been openly archived since the network's inception. The channel code `HHZ` selects the vertical-component broadband channel running at 100 sps.

FDSN identifiers follow the SEED standard. A network code is two characters. A station code is three to five. A location code is two characters or empty. A channel code is three characters whose first letter encodes the sampling rate band (H = high-band broadband, 80–250 sps), second letter encodes the instrument type (H = high-gain seismometer), and third letter encodes the component orientation (Z = vertical, N/E = horizontal).

The fetch window is one calendar day. That is the unit of work for a detection run: enough data to build a reliable STA/LTA threshold, short enough that you are not managing a multi-gigabyte download.

In [ ]:
# --- configurable parameters (instructor: edit here) ---

NETWORK  = 'GL'          # # verify: FDSN network code, EarthScope holdings
STATION  = 'ICESG'       # # verify: GLISN station on the Greenland ice sheet
LOCATION = '00'          # # verify: location code for this station
CHANNEL  = 'HHZ'         # vertical broadband, 100 sps

# Day of data to fetch.  Change this to any day the station was running.
T_START  = UTCDateTime('2020-07-15T00:00:00') if OBSPY_AVAILABLE else None
T_END    = UTCDateTime('2020-07-16T00:00:00') if OBSPY_AVAILABLE else None

SAVED_WAVEFORM = 'cryoseismicity_day.mseed'   # local cache after the first fetch
SAVED_INVENTORY = 'cryoseismicity_inv.xml'    # StationXML for response removal

print(f'Target: {NETWORK}.{STATION}.{LOCATION}.{CHANNEL}')
print(f'Window: {T_START} to {T_END}')
print(f'Nominal volume: {100 * 86400 * 4 / 1e6:.1f} MB (one component, 100 sps, 4 B/sample)')

In [ ]:
# NOT EXECUTED AT BUILD
# This cell downloads data from the EarthScope FDSN web service.
# It saves two files locally so subsequent cells can run offline.
# Expected download size: ~10 MB for one station-day at 100 sps.

import os

if not OBSPY_AVAILABLE:
    raise RuntimeError('Install obspy first.')

client = Client('IRIS')   # EarthScope FDSN endpoint (IRIS name retained for compatibility)

if not os.path.exists(SAVED_WAVEFORM):
    print('Fetching waveforms ...')
    st = client.get_waveforms(
        network=NETWORK, station=STATION, location=LOCATION,
        channel=CHANNEL, starttime=T_START, endtime=T_END,
    )
    st.write(SAVED_WAVEFORM, format='MSEED')
    print(f'  saved {SAVED_WAVEFORM} ({os.path.getsize(SAVED_WAVEFORM) / 1e6:.1f} MB)')
else:
    print(f'Using cached {SAVED_WAVEFORM}')

if not os.path.exists(SAVED_INVENTORY):
    print('Fetching station metadata (response) ...')
    inv = client.get_stations(
        network=NETWORK, station=STATION, location=LOCATION,
        channel=CHANNEL, starttime=T_START, endtime=T_END,
        level='response',
    )
    inv.write(SAVED_INVENTORY, format='STATIONXML')
    print(f'  saved {SAVED_INVENTORY}')
else:
    print(f'Using cached {SAVED_INVENTORY}')

## Raw counts, response removal, and bandpass

A freshly fetched waveform is raw digital counts. The number 4,294,967 means nothing until you know the sensitivity of the seismometer (volts per meter per second of ground velocity) and the gain of the digitizer (counts per volt). Those numbers are in the StationXML file. Deconvolving the full instrument response transfers the record from counts to physical ground velocity — or acceleration, if you prefer — and makes waveforms from different instruments comparable.

After response removal the record still contains the full seismic noise field, which is dominated at low frequencies (below about 0.1 Hz) by tidal and barometric loading, at 0.1–0.3 Hz by the secondary ocean microseism, and at higher frequencies by wind-induced noise and whatever process you are trying to detect. On ice there is an additional broadband background from basal water. A bandpass filter isolates the frequency band of interest. Icequakes from stick-slip basal sliding are typically richest between 1 and 10 Hz {cite}`podolskiy2016`; surface-fracture events (crevassing) often have energy extending to 30 Hz and above {cite}`aster2017`. The spectrogram of the full day, computed in the cell after next, will show you which band is cleanest on your particular day and station.

In [ ]:
# NOT EXECUTED AT BUILD
# Loads the saved MiniSEED, removes instrument response, bandpasses,
# and extracts the numpy arrays used by all later cells.

from obspy import read as obs_read
from obspy import read_inventory

st_raw = obs_read(SAVED_WAVEFORM)
inv    = read_inventory(SAVED_INVENTORY)

tr = st_raw[0].copy()
print(f'Trace: {tr.id}  |  {tr.stats.sampling_rate} sps  |  {tr.stats.npts} samples')
print(f'       starts {tr.stats.starttime}   ends {tr.stats.endtime}')

# Step 1: detrend (remove any linear drift)
tr.detrend('demean')
tr.detrend('linear')

# Step 2: remove instrument response
# pre_filt suppresses noise below 0.5 Hz and above Nyquist during deconvolution.
pre_filt = (0.005, 0.01, 45.0, 50.0)   # four corners of the cosine taper, Hz
tr.remove_response(inventory=inv, output='VEL', pre_filt=pre_filt, water_level=60)

# Step 3: bandpass to the icequake band
BP_LOW  = 1.0    # Hz — below this: ocean microseism, tidal noise
BP_HIGH = 30.0   # Hz — above this: instrument self-noise and aliasing artifacts
tr.filter('bandpass', freqmin=BP_LOW, freqmax=BP_HIGH, corners=4, zerophase=True)

# Pull out the numpy array and the sample rate for later cells
DATA  = tr.data.astype(np.float64)    # ground velocity, m/s
FS    = tr.stats.sampling_rate        # samples per second
TIMES = np.arange(len(DATA)) / FS     # seconds from start of day

print(f'\nBandpassed trace: {len(DATA)} samples, {len(DATA)/FS/3600:.2f} hours')
print(f'RMS velocity: {np.sqrt(np.mean(DATA**2))*1e9:.1f} nm/s')

## The noise field

Before searching for events, look at the full day. The spectrogram below spreads time along the horizontal axis and frequency along the vertical, with color encoding energy density. Three features are worth picking out before you look at it.

At 0.1–0.3 Hz the ocean microseism — generated by opposing ocean waves interfering to produce a standing pressure wave on the seafloor — arrives everywhere on Earth as a nearly constant hum. On a polar station it is one of the loudest signals on the record and, for our purposes, irrelevant noise. The bandpass you applied earlier attenuates most of it.

In the 1–10 Hz band you may see diurnal pulsing, louder in the afternoon than at night, driven by surface melt and the seismic noise it produces as water moves through cracks and conduits. You may also see brief transients, elevated energy in narrow frequency bands lasting a fraction of a second, which are the icequakes. A human eye can pick them out on the spectrogram for loud events; the STA/LTA detector in the next section does the same thing systematically for events the eye misses.

In [ ]:
# This cell runs offline once DATA, FS, TIMES are defined above.
# If running the lab without fetching data, substitute a synthetic trace:
#   rng = np.random.default_rng(7); FS = 100.0
#   DATA = rng.normal(0, 1e-7, int(FS * 3600 * 6))
#   TIMES = np.arange(len(DATA)) / FS

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                         gridspec_kw={'height_ratios': [1, 2]})

# Top panel: waveform (first 6 hours for readability)
six_hr = int(FS * 6 * 3600)
t6 = TIMES[:six_hr] / 3600
axes[0].plot(t6, DATA[:six_hr] * 1e9, lw=0.4, color='steelblue')
axes[0].set_ylabel('velocity (nm/s)')
axes[0].set_title(f'{NETWORK}.{STATION} — bandpass {BP_LOW}–{BP_HIGH} Hz')

# Bottom panel: spectrogram of the same 6-hour window
nperseg = int(FS * 2)                          # 2-second windows
noverlap = int(nperseg * 0.9)                  # 90 % overlap
f, t_seg, Sxx = spectrogram(DATA[:six_hr], fs=FS,
                             nperseg=nperseg, noverlap=noverlap)

# Show only 0.5–40 Hz where the action is
freq_mask = (f >= 0.5) & (f <= 40.0)
axes[1].pcolormesh(t_seg / 3600, f[freq_mask],
                   10 * np.log10(Sxx[freq_mask] + 1e-30),
                   shading='gouraud', cmap='inferno', vmin=-180, vmax=-120)
axes[1].set_ylabel('frequency (Hz)')
axes[1].set_xlabel('time (hours from start of day)')
axes[1].set_ylim(0.5, 40)

plt.tight_layout()
plt.show()

## The STA/LTA detector

The chapter describes the STA/LTA method in one sentence: watch for a jump in the seismogram's short-term energy against its long-term background. Here we derive and implement it step by step so the tuning parameters have physical meaning.

The **long-term average** (LTA) estimates the ambient noise level at a given moment. It is the mean squared amplitude of the bandpassed signal over a trailing window of typically 30–60 seconds — long enough to average over many microseismic cycles but short enough to track slow drifts in the noise floor across the day.

The **short-term average** (STA) does the same over a window of typically 0.5–2 seconds — comparable to the duration of an icequake. When a sudden event arrives, STA spikes while LTA barely moves. Their ratio

$$R(t) = \frac{\text{STA}(t)}{\text{LTA}(t)}$$

sits near 1 in quiet periods and jumps well above 1 when an event arrives. A trigger is declared when $R$ exceeds a threshold $\theta_{\text{on}}$, and it is held until $R$ falls below a lower threshold $\theta_{\text{off}}$ (the hysteresis prevents one long event from being sliced into many short ones). The three free parameters — STA window $w_s$, LTA window $w_l$, and the on/off thresholds — set the sensitivity and false-alarm rate of the detector and must be tuned on a representative stretch of data.

The implementation below works entirely in numpy. The key operation is a **running mean of squared amplitude**, computed efficiently with `np.cumsum` rather than a loop.

In [ ]:
def running_mean_sq(x, n):
    """Running mean of x**2 over a trailing window of n samples.

    Uses the cumulative-sum trick so the cost is O(N) regardless of n.
    The first n-1 samples are set to the overall mean squared amplitude
    to avoid dividing by very small numbers at the trace start.
    """
    x2 = x ** 2
    cs = np.cumsum(x2)
    out = np.empty_like(x2)
    out[:n] = cs[n - 1] / n            # fill short prefix with first full mean
    out[n:] = (cs[n:] - cs[:-n]) / n  # sliding window
    return out


def stalta(data, fs, sta_s=1.0, lta_s=30.0):
    """Compute the STA/LTA characteristic function.

    Parameters
    ----------
    data  : 1-D array of ground velocity (m/s), uniform sampling
    fs    : sample rate (Hz)
    sta_s : short-term window length (seconds)
    lta_s : long-term window length (seconds)

    Returns
    -------
    ratio : STA/LTA array, same length as data
    """
    n_sta = max(1, int(sta_s * fs))
    n_lta = max(1, int(lta_s * fs))
    sta   = running_mean_sq(data, n_sta)
    lta   = running_mean_sq(data, n_lta)
    ratio = np.where(lta > 0, sta / lta, 0.0)
    return ratio


def trigger(ratio, fs, threshold_on=3.0, threshold_off=1.5, min_gap_s=1.0):
    """Extract trigger-on/off intervals from a STA/LTA ratio array.

    Returns a list of (t_on, t_off) pairs in seconds from the trace start.
    Events separated by less than min_gap_s are merged into one.
    """
    on_samples  = []
    off_samples = []
    above = False
    for i, r in enumerate(ratio):
        if not above and r >= threshold_on:
            above = True
            on_samples.append(i)
        elif above and r < threshold_off:
            above = False
            off_samples.append(i)
    if above:
        off_samples.append(len(ratio) - 1)

    events = list(zip(np.array(on_samples) / fs,
                      np.array(off_samples) / fs))

    # merge events that are too close together
    merged = []
    for ev in events:
        if merged and ev[0] - merged[-1][1] < min_gap_s:
            merged[-1] = (merged[-1][0], ev[1])
        else:
            merged.append(ev)
    return merged


print('STA/LTA functions defined.')
print('  running_mean_sq : O(N) running window via cumsum')
print('  stalta          : returns ratio array')
print('  trigger         : returns list of (t_on, t_off) in seconds')

## Tuning and running the detector

The next cell runs the detector on the full day with a set of default parameters and plots the STA/LTA ratio alongside the raw waveform so you can see which peaks correspond to plausible events. Tuning is an iterative process: lower $\theta_{\text{on}}$ catches more events but admits more false positives; longer STA windows average over short impulsive events and raise the detection threshold for them; shorter LTA windows make the baseline too responsive to the events themselves, which inflates LTA and suppresses the ratio right after a cluster.

A practical first pass is to pick a two-hour window that the spectrogram showed as moderately active, run the detector with the defaults, and visually verify five or ten of the triggers against the waveform. If most of the flagged windows contain an obvious impulsive arrival, the parameters are usable; if they mostly flag diffuse noise, raise $\theta_{\text{on}}$.

In [ ]:
# Default STA/LTA parameters — adjust for Task 1 (see Tasks cell below)
STA_S         = 1.0    # short-term window, seconds
LTA_S         = 30.0   # long-term window, seconds
THRESH_ON     = 3.5    # trigger-on threshold
THRESH_OFF    = 1.5    # trigger-off threshold
MIN_GAP_S     = 2.0    # minimum gap between separate events, seconds

ratio = stalta(DATA, FS, sta_s=STA_S, lta_s=LTA_S)
events = trigger(ratio, FS,
                 threshold_on=THRESH_ON, threshold_off=THRESH_OFF,
                 min_gap_s=MIN_GAP_S)

print(f'Detected {len(events)} events with STA={STA_S} s, LTA={LTA_S} s, '
      f'on/off={THRESH_ON}/{THRESH_OFF}')

# Plot the first 6 hours of the ratio and waveform together
six_hr = int(FS * 6 * 3600)
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(TIMES[:six_hr] / 3600, DATA[:six_hr] * 1e9,
             lw=0.3, color='steelblue')
axes[0].set_ylabel('velocity (nm/s)')

axes[1].plot(TIMES[:six_hr] / 3600, ratio[:six_hr], lw=0.5, color='k')
axes[1].axhline(THRESH_ON,  color='crimson', ls='--', lw=1, label=f'on = {THRESH_ON}')
axes[1].axhline(THRESH_OFF, color='orange',  ls=':', lw=1, label=f'off = {THRESH_OFF}')
# shade detected windows
for t_on, t_off in events:
    if t_off < 6 * 3600:
        axes[1].axvspan(t_on / 3600, t_off / 3600, alpha=0.15, color='crimson')
axes[1].set_ylabel('STA/LTA ratio')
axes[1].set_xlabel('time (hours from start of day)')
axes[1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## Classifying detections

Not all triggers are icequakes, and not all icequakes are the same. The chapter distinguishes three main families by a combination of frequency content and duration {cite}`podolskiy2016`.

**Basal stick-slip events** are short (0.1–2 seconds), energetic, and band-limited in the 1–10 Hz range. The rupture is a small area of the bed unlocking, radiating a pulse that looks like a miniature tectonic earthquake — a sharp onset, a brief coda, an abrupt end. Spectrally they appear as a broadband spike that decays smoothly with frequency above the corner frequency.

**Surface and englacial fracture events** (crevassing) are also impulsive but tend to be shorter (under 0.5 seconds) and richer in high-frequency content (5–40 Hz) because the source is a crack propagating through cold, stiff ice rather than a sliding contact. Their spectra are flatter than basal events across the 5–30 Hz band {cite}`aster2017`.

**Icequake tremor** from subglacial water flow is neither impulsive nor discrete. It shows up as sustained elevated energy in a narrow frequency band (typically 1–5 Hz), persisting for minutes to hours, modulated by melt input. STA/LTA does not find it cleanly; spectral tracking is more appropriate.

The two diagnostics we compute for each detected event are its **dominant frequency** (peak of the power spectrum within the event window) and its **duration** (the t_on to t_off interval). In the 2-D space of (dominant frequency, duration), basal and surface events form separable clouds for a dataset where both are present.

In [ ]:
def characterise(data, fs, t_on, t_off, pad_s=0.5):
    """Extract dominant frequency and duration for one detected window.

    pad_s extends the window on each side (up to the trace boundary)
    to include the event onset and tail without being contaminated by
    the adjacent noise if the STA/LTA window was too tight.
    """
    i0 = max(0, int((t_on  - pad_s) * fs))
    i1 = min(len(data), int((t_off + pad_s) * fs))
    snippet = data[i0:i1]
    if len(snippet) < 4:
        return np.nan, np.nan
    n_fft = max(256, 2 ** int(np.ceil(np.log2(len(snippet)))))
    spectrum = np.abs(np.fft.rfft(snippet * np.hanning(len(snippet)), n=n_fft)) ** 2
    freqs    = np.fft.rfftfreq(n_fft, d=1.0 / fs)
    # restrict to 1–40 Hz
    band = (freqs >= 1.0) & (freqs <= 40.0)
    dominant_freq = freqs[band][np.argmax(spectrum[band])]
    duration      = t_off - t_on
    return dominant_freq, duration


dom_freqs  = []
durations  = []
for t_on, t_off in events:
    f_dom, dur = characterise(DATA, FS, t_on, t_off)
    dom_freqs.append(f_dom)
    durations.append(dur)

dom_freqs = np.array(dom_freqs)
durations = np.array(durations)

# Classification heuristic: basal = low dominant frequency AND longer duration
basal   = (dom_freqs <  8.0) & (durations > 0.3)
surface = (dom_freqs >= 8.0) | (durations <= 0.3)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(dom_freqs[basal],   durations[basal],   s=20, c='royalblue',
           alpha=0.7, label=f'basal-type ({basal.sum()})')
ax.scatter(dom_freqs[surface], durations[surface], s=20, c='firebrick',
           alpha=0.7, label=f'surface-type ({surface.sum()})')
ax.set_xlabel('dominant frequency (Hz)')
ax.set_ylabel('duration (s)')
ax.set_title('Event classification by frequency and duration')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Total events detected: {len(events)}')
print(f'  basal-type:   {basal.sum()}  (dom. freq < 8 Hz, dur > 0.3 s)')
print(f'  surface-type: {surface.sum()}  (dom. freq >= 8 Hz or dur <= 0.3 s)')

## Stacking detected waveforms

Individual icequake waveforms are noisy. Stacking many events aligned on their trigger onset averages out the incoherent noise and leaves the common waveform shape, which is the coherent part of the signal: the P-wave onset, the peak ground motion, and the decay of the coda. If the events are truly from a repeating source (a patch of bed that sticks and slips repeatedly at the same location), the stack will be clean; if the detections are a mixture of unrelated sources, the stack will be broad and smeared.

The cell below stacks both the basal-type and surface-type events separately. It normalises each snippet to unit peak amplitude before stacking so that the largest events do not dominate, and uses the trigger-on time as the alignment point. The asymmetry of the stacked waveform — whether the onset is sharper than the coda, or vice versa — is a qualitative indicator of source complexity.

In [ ]:
def stack_events(data, fs, times_on, half_win_s=2.0):
    """Stack event snippets centred on their onset times.

    Each snippet is normalised to unit peak absolute amplitude before
    stacking.  Returns the stacked trace and a time axis centred on 0.
    """
    n_half = int(half_win_s * fs)
    n_win  = 2 * n_half
    stack  = np.zeros(n_win)
    count  = 0
    for t_on in times_on:
        i0 = int(t_on * fs) - n_half
        i1 = i0 + n_win
        if i0 < 0 or i1 >= len(data):
            continue
        snippet = data[i0:i1].copy()
        peak    = np.max(np.abs(snippet))
        if peak > 0:
            stack += snippet / peak
            count += 1
    if count > 0:
        stack /= count
    t_axis = (np.arange(n_win) - n_half) / fs
    return t_axis, stack, count


events_arr = np.array(events)
t_on_basal   = events_arr[basal,   0] if basal.any()   else np.array([])
t_on_surface = events_arr[surface, 0] if surface.any() else np.array([])

t_ax_b, stk_b, n_b = stack_events(DATA, FS, t_on_basal)
t_ax_s, stk_s, n_s = stack_events(DATA, FS, t_on_surface)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].plot(t_ax_b, stk_b, lw=1, color='royalblue')
axes[0].axvline(0, color='gray', lw=0.8, ls='--')
axes[0].set_title(f'Basal-type stack (N = {n_b})')
axes[0].set_xlabel('time relative to trigger onset (s)')
axes[0].set_ylabel('normalised amplitude')

axes[1].plot(t_ax_s, stk_s, lw=1, color='firebrick')
axes[1].axvline(0, color='gray', lw=0.8, ls='--')
axes[1].set_title(f'Surface-type stack (N = {n_s})')
axes[1].set_xlabel('time relative to trigger onset (s)')

plt.tight_layout()
plt.show()

## Limitations of a single station

Everything so far has been done with one seismometer. That is a real limitation. A single station tells you *when* an event occurred, its rough frequency content, and its amplitude (which depends on distance in an unknown way). It does not tell you *where* the event occurred.

Location requires at least three stations and a model for how fast seismic waves travel through the medium. The standard approach compares the arrival times of the P-wave (the compressional wave, faster) and S-wave (the shear wave, slower) at each station. The difference in arrival times is proportional to the distance from the station to the source, so three or more such differences over-determine the source coordinates. Alternatively, on an array of closely-spaced sensors, the apparent slowness and azimuth of a wavefront can be read from the time delays between adjacent sensors (beamforming), which is how the large-aperture Greenland and Antarctic arrays are routinely used.

The distinction between basal and surface events — which we made here on frequency and duration alone — would be far sharper with even a second sensor a kilometer away: a basal event at 800 meters depth has a longer source-to-sensor path than a surface event at the same epicentre, so its P-wave arrives later relative to the S-wave, and its high-frequency content is attenuated more by the ice. Travel-time location is the natural extension of this single-station lab, and {doc}`cryoseismicity` describes how it is done in practice.

## Tasks

**Task 1 — STA/LTA parameter sensitivity.**

Run the detector three times on the same day of data, varying one parameter at a time while holding the others at the defaults:

- (a) STA window: 0.5 s, 1.0 s (default), 3.0 s. Print the event count for each. For which STA length do you expect the most false positives, and why?
- (b) Trigger-on threshold: 2.5, 3.5 (default), 5.0. Plot the ratio curve for a 30-minute window that contains several events, and shade the windows detected by each threshold. Which threshold misses small but real-looking events, and which floods the catalogue with noise?
- (c) LTA window: 10 s, 30 s (default), 120 s. Think about what happens when a cluster of events arrives in rapid succession. A short LTA rises after the first event in a cluster, suppressing the ratio for subsequent events. Plot the ratio for a clustered stretch and count how many events the three LTA windows find.

Fill in the code blocks and write two sentences below each sub-task explaining the pattern you see in physical terms.

**Task 2 — Diurnal statistics of detections versus melt.**

Using the full day's catalogue, bin the detection times into one-hour intervals and plot the hourly event count as a histogram. Most polar seismic studies show a diurnal cycle in surface-fracture events, peaking in the afternoon when surface melting is highest and crevasses are being hydraulically loaded. Basal events, if driven by effective-pressure changes at the bed rather than by direct insolation, should show a phase-lagged peak or no clear diurnal signal at all.

- (a) Make separate histograms for basal-type and surface-type events and compare their peak times.
- (b) A rough surface energy proxy is available without auxiliary data: the broadband RMS amplitude of the seismogram in a low-frequency band (0.1–0.5 Hz, below the icequake band) tracks the seismic noise generated by surface meltwater. Compute the hourly median RMS in that band and overlay it on the histograms (normalised to fit the same y-axis).
- (c) Write a short paragraph — three to five sentences — interpreting the phase relationships between melt proxy, surface-fracture events, and basal events. What lag, if any, is consistent with water percolating from the surface to the bed before affecting basal sliding?

In [ ]:
# Task 1 scaffolding

# (a) Vary STA window
for sta_test in [0.5, 1.0, 3.0]:
    r_test = stalta(DATA, FS, sta_s=sta_test, lta_s=LTA_S)
    ev_test = trigger(r_test, FS, threshold_on=THRESH_ON,
                      threshold_off=THRESH_OFF, min_gap_s=MIN_GAP_S)
    # YOUR CODE HERE: print(f'STA = {sta_test} s  ->  {len(ev_test)} events')
    pass

# (b) Vary trigger-on threshold — fill in the plot
fig, ax = plt.subplots(figsize=(10, 3))
window_slice = slice(int(FS * 3 * 3600), int(FS * 3.5 * 3600))   # 30-min window
t_win = TIMES[window_slice] / 3600
ax.plot(t_win, ratio[window_slice], 'k', lw=0.7)
for thresh, color in [(2.5, 'green'), (3.5, 'orange'), (5.0, 'red')]:
    ax.axhline(thresh, color=color, ls='--', lw=1, label=f'on = {thresh}')
    # YOUR CODE HERE: shade detected windows for this threshold
ax.set_xlabel('time (hours)')
ax.set_ylabel('STA/LTA')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Task 2 scaffolding
# Bin detection times into hourly counts for basal vs surface
t_on_all = np.array([e[0] for e in events])
hours_all = t_on_all / 3600.0

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.arange(0, 25, 1)
# YOUR CODE HERE:
# ax.hist(hours_all[basal],   bins=bins, alpha=0.6, label='basal-type',   color='royalblue')
# ax.hist(hours_all[surface], bins=bins, alpha=0.6, label='surface-type', color='firebrick')
ax.set_xlabel('hour of day')
ax.set_ylabel('events per hour')
ax.set_title('Diurnal detection histogram')
ax.legend()
plt.tight_layout()
plt.show()

## Synthesis

The catalogue of icequakes you built here is an underused constraint in glacier physics. The friction laws of {doc}`../thermomechanics/basal-motion` model basal sliding as a continuous process with a velocity determined by effective pressure, ice thickness, and bed roughness. But the seismology shows that at least some glaciers slide in discrete stick-slip episodes rather than continuously, and the rate and magnitude of those episodes carry information about the effective pressure and the frictional state of the bed — information that a surface velocity measurement alone cannot provide, because a glacier sliding jerkily at the right average rate looks identical to one sliding smoothly. The satellite-era surface-velocity records from {doc}`elevation` are rich but blind to this distinction; an icequake catalogue is the complementary dataset.

The connection to friction is not just conceptual. The inter-event time distribution of stick-slip seismicity follows a rate-and-state friction model in laboratory experiments {cite}`podolskiy2016`, and field catalogues from Greenland and Antarctica show the same scaling. If you could measure the moment of each slip event — requiring a calibrated seismometer and a source model, not just a detection — you could invert for the stress drop and the slip area, which are directly the quantities that appear in the Coulomb failure criterion and the hard-bed friction relations of {doc}`../thermomechanics/basal-motion`. The daily rate of events, even without moment estimation, tracks effective pressure changes driven by the subglacial drainage reorganisations described in {doc}`../thermomechanics/hydrology`.

Write two paragraphs addressing the following.

1. Your Task 2 analysis tested whether surface-fracture and basal events follow different diurnal rhythms. Describe, mechanistically, how surface meltwater reaching the bed through a crevasse system would change the effective pressure, the sliding velocity, and the expected rate of basal stick-slip events. What lag do you predict between the peak of surface energy and the peak of basal seismicity, and does your histogram match that prediction?

2. A single station cannot locate events, so the catalogue you built is a time series of detections, not a spatial map. Suppose you had a three-station array with 2-km spacing. Sketch (in words or a diagram) how you would use P-wave arrival-time differences to locate one of the basal events you detected. What additional information would let you distinguish a bed-level source from a source at 200 m depth? And why would even a rough location map — knowing which part of the glacier bed is seismically active — be valuable for constraining the friction-law parameters that the surface velocity record alone cannot provide?